# MLnode GAN Training Notebook

Train a **DCGAN-style GAN** on MNIST where *both* networks are MLnode JSON graphs:

```
Generator  gan_generator.mlnode.json      z (B, 100) → fake image (B, 1, 28, 28) in [-1, 1]
   Linear 100→6272 → Reshape (B,128,7,7) → BN+ReLU
   → ConvTranspose2d 128→64 (14×14) → BN+ReLU
   → ConvTranspose2d 64→1  (28×28) → Tanh

Discriminator  gan_discriminator.mlnode.json   image → real/fake logit (B, 1)
   Conv2d 1→64 s2 → LeakyReLU(0.2)
   → Conv2d 64→128 s2 → BN → LeakyReLU(0.2) → Dropout
   → Flatten → Linear 6272→1
```

The discriminator outputs a raw logit (no Sigmoid node) — we train with
`BCEWithLogitsLoss`, which is numerically safer.

In [ ]:
# Install MLnode (run once)
# !pip install -e /path/to/mlnode
# Or if already installed:
from mlnode import MLnodeModel
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Load Both Model Graphs

In [ ]:
G_model = MLnodeModel.load("gan_generator.mlnode.json")
D_model = MLnodeModel.load("gan_discriminator.mlnode.json")
print(G_model)
print(D_model)

## 2. Validate Both Graphs Statically

In [ ]:
for name, m in (("Generator", G_model), ("Discriminator", D_model)):
    result = m.validate()
    print(f"{name}: valid={result.valid}")
    for node_id, shape in result.node_shapes.items():
        print(f"  {node_id:12s} → {shape}")
    for err in result.errors:
        print(f"  ERROR: {err.message}")
    print()

## 3. Build the PyTorch Modules

In [ ]:
LATENT_DIM = 100

G = G_model.build().to(device)
D = D_model.build().to(device)

z = torch.randn(4, LATENT_DIM, device=device)
fake = G(z)
logit = D(fake)
print(f"z:       {z.shape}")
print(f"G(z):    {fake.shape}  (range [{fake.min():.2f}, {fake.max():.2f}])")
print(f"D(G(z)): {logit.shape}")
print(f"G params: {sum(p.numel() for p in G.parameters()):,}")
print(f"D params: {sum(p.numel() for p in D.parameters()):,}")

## 4. Data Pipeline

Images are normalized to **[-1, 1]** to match the generator's `Tanh` output.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),  # [0,1] → [-1,1]
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
print(f"Training samples: {len(train_dataset):,} | batches/epoch: {len(train_loader)}")

## 5. Adversarial Training

Standard GAN alternation, per batch:

1. **D step** — maximize log D(x) + log(1 − D(G(z))): real images labeled 1, fakes (detached) labeled 0.
2. **G step** — non-saturating trick: minimize −log D(G(z)) (fakes labeled 1).


In [ ]:
EPOCHS = 5
LR = 2e-4

criterion = nn.BCEWithLogitsLoss()
optG = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
optD = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

# Fixed noise → watch the same 64 samples evolve across epochs
fixed_noise = torch.randn(64, LATENT_DIM, device=device)

real_label, fake_label = 1.0, 0.0
print(f"Adam lr={LR}, betas=(0.5, 0.999) | epochs={EPOCHS}")

In [ ]:
history = {"d_loss": [], "g_loss": []}
snapshots = []  # fixed-noise samples after each epoch

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    d_running, g_running = 0.0, 0.0

    for i, (real, _) in enumerate(train_loader):
        real = real.to(device)
        b = real.size(0)
        ones = torch.full((b, 1), real_label, device=device)
        zeros = torch.full((b, 1), fake_label, device=device)

        # ── Discriminator step ──
        optD.zero_grad()
        loss_real = criterion(D(real), ones)
        fake = G(torch.randn(b, LATENT_DIM, device=device))
        loss_fake = criterion(D(fake.detach()), zeros)
        d_loss = loss_real + loss_fake
        d_loss.backward()
        optD.step()

        # ── Generator step ──
        optG.zero_grad()
        g_loss = criterion(D(fake), ones)  # non-saturating
        g_loss.backward()
        optG.step()

        d_running += d_loss.item()
        g_running += g_loss.item()
        if i % 100 == 0:
            print(f"  epoch {epoch} [{i:>3}/{len(train_loader)}] D={d_loss.item():.3f} G={g_loss.item():.3f}")

    history["d_loss"].append(d_running / len(train_loader))
    history["g_loss"].append(g_running / len(train_loader))

    G.eval()
    with torch.no_grad():
        snapshots.append(G(fixed_noise).cpu())
    G.train()
    print(f"Epoch {epoch}: D={history['d_loss'][-1]:.3f} G={history['g_loss'][-1]:.3f} | {time.time()-t0:.1f}s")

## 6. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
epochs = range(1, len(history["d_loss"]) + 1)
plt.plot(epochs, history["d_loss"], label="Discriminator")
plt.plot(epochs, history["g_loss"], label="Generator")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("GAN training losses")
plt.tight_layout()
plt.show()

## 7. Generated Samples

The same 64 latent vectors, after the final epoch (values mapped back from [-1, 1] to [0, 1]).

In [ ]:
from torchvision.utils import make_grid

grid = make_grid(snapshots[-1], nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.axis("off")
plt.title(f"Fixed-noise samples after epoch {len(snapshots)}")
plt.show()

# Evolution across epochs (first fixed-noise sample per epoch)
fig, axes = plt.subplots(1, len(snapshots), figsize=(2 * len(snapshots), 2.2), squeeze=False)
for ax, snap, ep in zip(axes[0], snapshots, range(1, len(snapshots) + 1)):
    ax.imshow(snap[0].squeeze(), cmap="gray")
    ax.set_title(f"epoch {ep}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Save the Models

In [ ]:
torch.save(G.state_dict(), "gan_generator_weights.pth")
torch.save(D.state_dict(), "gan_discriminator_weights.pth")
print("Saved generator and discriminator weights.")
print("Architectures live in gan_generator.mlnode.json / gan_discriminator.mlnode.json —")
print("open them in mlnode-editor to inspect or modify the graphs visually.")